# Question 02 Scatterplot Planning (Scatterplot of Health vs. Education)

## Step 01: Load Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean/panel_1980_2010.csv")

print(df.shape)
print(df.columns.tolist())

(615, 19)
['Country Name', 'Country Code', 'Year', 'region', 'income', 'infant_mortality', 'life_expectancy', 'under5_mortality', 'literacy_rate', 'primary_enrollment', 'prog_to_secondary', 'secondary_enrollment', 'adolescent_fertility_per_1000_women', 'open_defecation_pct', 'sanitation_basic_pct', 'sanitation_safely_managed_pct', 'water_basic_pct', 'gdp_per_capita_PPP', 'gdp_per_capita_const2010usd']


## Step 02: Define Variable Groups (health vs. education; makes dropdowns easier to implement)

In [3]:
education_vars = [
    "literacy_rate",
    "primary_enrollment",
    "prog_to_secondary",    # not-very-good coverage
    "secondary_enrollment", # mostly-reliable coverage (1980-2020), but may have missing values for highly-developed countries
]

# below: all very good coverage 1980-2020
health_vars = [
    "life_expectancy",
    "infant_mortality",
    "under5_mortality",
]

## Step 03 (preliminary) - start with a single year for planning (we'll move to a dynamic year-slider later)

In [8]:
# 3.1. start with year = 2004
year = 2002

# 3.2. get data for all countries in only 2000 (as a stepping stone to eventual final product)
plot_df = df[df["Year"] == year].copy()

print(plot_df.shape)

(15, 19)


## Step 04 - first non-interactive scatterplot (preliminary baseline)

In [11]:
import plotly.express as px

# 4.1. Plot first scatterplot (of secondary enrollment versus under5 mortality)
fig = px.scatter(
    plot_df,
    x="secondary_enrollment",
    y="under5_mortality",
    color="income",
    hover_name="Country Name",
)

fig.show()

## Step 05-06 - Improving the Static Chart in Step 4 (post-baseline, pre-interaction)

Descriptive Tooltips, Trendline, Correlation comp+classify

1. Tooltips: (see country name, region, income level, and both x and y)
2. Trendline
3. Correlation (value and classification (weak positive, etc.))

*Note: temporarily: x="secondary_enrollment" and y="under5_mortality"*

In [17]:
# May need to run this install the first time
# %pip install statsmodels

import statsmodels.api as sm # for trendline

# 5.1. Plot first scatterplot (of secondary enrollment versus under5 mortality)
fig = px.scatter(
    plot_df,
    x="secondary_enrollment",   # TODO: make a user-selectable parameter
    y="under5_mortality",       # TODO: make a user-selectable parameter
    color="income",
    hover_name="Country Name",
    hover_data=[
        "region",
        "income",
        "life_expectancy",
        "under5_mortality"
    ],
    # trendline="ols"
)

# 5.3. Fit one regression line across all displayed countries
trend_df = plot_df[["secondary_enrollment", "under5_mortality"]].dropna()
x = trend_df["secondary_enrollment"]
y = trend_df["under5_mortality"]

# 5.4. create OLS regression
X_model = sm.add_constant(x)
model = sm.OLS(y, X_model).fit()
# print(model.summary())  # summarise model

# 5.5. Plot line: Generate x-range
x_line = np.linspace(
    x.min(),
    x.max(),
    100 # multiply decimals by 100 to get percents
)

# 5.6. Generate y-predictions
X_line = sm.add_constant(x_line)    # add bias (y-intercept)
y_pred = model.predict(X_line)

# 5.7. Add the all-country trendline (includes all income levels) to the scatterplot
fig.add_scatter(
    x=x_line,
    y=y_pred,
    mode="lines",
    name="Overall Trend",
    line=dict(
        color="gray",
        dash="dash",
        width=2,
    )
)

fig.show()

## Step 7-8. Compute Correlation via NumPy, then classify it (a signed scalar)

In [ ]:
# 7. Compute correlation
corr_df = plot_df[  # select education and sanitation columns
    ["secondary_enrollment", "under5_mortality"]
].dropna()

correlation_value = corr_df.corr().iloc[0, 1]

print(f"Correlation = {correlation_value:.2f}")


# 8. Classify correlation
def classify_correlation(r):

    magnitude = abs(r)

    if magnitude < 0.2:
        return "Very Weak"

    elif magnitude < 0.4:
        return "Weak"

    elif magnitude < 0.6:
        return "Moderate"

    elif magnitude < 0.8:
        return "Strong"

    return "Very Strong"

# Usage of classify correlation example:
label = classify_correlation(correlation_value)
print(label)

Correlation = -0.87
Very Strong


# Steps 9-16 - Building-Blocks for Interactivity 

## Step 9. Year-Selection Slider Widget

In [2]:
# 9.1. may have to run this install command at first
# %pip install ipywidgets

import ipywidgets as widgets

# 9.2. add year-slider widget
year_slider = widgets.IntSlider(
    value=2000,     # default position
    min=1980,
    max=2020,
    step=1,
    description="Year"
)

# 9.3. Dev-time inspect
year_slider

IntSlider(value=2000, description='Year', max=2020, min=1980)

## Step 10-11. Year-Selection Textbox + Sync selector and textbox

In [ ]:
# 10. Year selection textbox
year_text = widgets.BoundedIntText(
    value=1990,
    min=1980,
    max=2020,
    description="Year"
)

# 11. Link them together. One update will update another. 
widgets.jslink(
    (year_slider, "value"),
    (year_text, "value")
)

# year_slider
year_text   # testing - type a year and press enter. The previous cell's slider should jump in sync. 

BoundedIntText(value=1990, description='Year', max=2020, min=1980)